In [8]:
import os
import glob
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from scipy.ndimage import median_filter
import xgboost as xgb
import librosa

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
SMILE_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
CONFIG_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\spec_pitch.conf"

OPENSMILE_TMP = "opensmile_tmp"
OUT_TRAIN = "csv_train_mod"
OUT_TEST = "csv_test_mod"

os.makedirs(OPENSMILE_TMP, exist_ok=True)
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)

SR = 16000
WIN_DUR = 0.100
HOP_DUR = 0.010
FREQ_MAX = 2000
SMOOTH_WIN = 7   # frames (~70 ms)

# =============================================================================
# FIND ALL .wav/.txt PAIRS (ORDER PRESERVED)
# =============================================================================
wav_files = sorted(glob.glob(os.path.join(BASE_DIR, "*.wav")))
txt_files = [f.replace(".wav", "_label_audacity.txt") for f in wav_files]

valid_pairs = [
    (wav, txt) for wav, txt in zip(wav_files, txt_files)
    if os.path.exists(txt)
]

print(f"Found {len(valid_pairs)} valid .wav/.txt pairs")

train_pairs = valid_pairs[:80]
test_pairs = valid_pairs[80:100]

# =============================================================================
# LABEL PARSING
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if p[2] == "Wheeze" else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels

def wheeze_priority_label(labels, t_start, t_end):
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0

# =============================================================================
# FFT FEATURE EXTRACTION
# =============================================================================
def extract_fft_features(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []

    for start in range(0, len(audio) - win_len + 1, hop_len):
        end = start + win_len
        frame = audio[start:end] * np.hanning(win_len)

        X = np.fft.rfft(frame)
        freqs = np.fft.rfftfreq(win_len, 1 / SR)
        mag = np.abs(X)
        mag[0] = 0

        valid = freqs <= FREQ_MAX
        freqs, mag = freqs[valid], mag[valid]

        idx = np.argmax(mag)

        t_start = start / SR
        t_end = end / SR

        rows.append([
            Path(wav_path).name,
            t_start,
            mag[idx],
            freqs[idx],
            wheeze_priority_label(labels, t_start, t_end)
        ])

    return pd.DataFrame(rows, columns=[
        "file", "frameTime", "fft_amplitude", "fft_frequency", "label"
    ])

# =============================================================================
# OPENSMILE EXTRACTION + PARSING (YOUR LOGIC)
# =============================================================================
def extract_opensmile_features(wav_file, txt_file):
    base_name = Path(wav_file).stem
    output_csv = os.path.join(OPENSMILE_TMP, f"{base_name}_energy.csv")

    cmd = [SMILE_PATH, "-C", CONFIG_PATH, "-I", wav_file, "-O", output_csv]
    subprocess.run(cmd, check=True, capture_output=True)

    df = pd.read_csv(output_csv, header=None)
    df_split = df[0].str.split(";", expand=True)
    df_split.columns = [
        "frameIndex",
        "frameTime",
        "spectralFlux_log",
        "spectralCentroid_log",
        "spectralMaxPos_log",
        "spectralMinPos_log",
        "pcm_RMSenergy",
        "pcm_LOGenergy",
        "F0direction",
        "directionScore"
    ]

    df_split["frameTime"] = pd.to_numeric(df_split["frameTime"], errors="coerce")
    df_split = df_split[df_split["frameTime"].notna()].reset_index(drop=True)
    df_split["file"] = Path(wav_file).name

    return df_split

# =============================================================================
# MERGE FFT + OPENSMILE
# =============================================================================
def extract_and_merge(wav, txt):
    fft_df = extract_fft_features(wav, txt)
    os_df = extract_opensmile_features(wav, txt)

    fft_df["frameTime"] = fft_df["frameTime"].round(3)
    os_df["frameTime"] = os_df["frameTime"].round(3)

    return pd.merge(
        fft_df,
        os_df,
        on=["file", "frameTime"],
        how="left"
    )

# =============================================================================
# PROCESS DATASETS
# =============================================================================
def process_pairs(pairs, out_dir, tag):
    dfs = []
    print(f"\nProcessing {len(pairs)} {tag} files...")
    for wav, txt in tqdm(pairs):
        df = extract_and_merge(wav, txt)
        df.to_csv(os.path.join(out_dir, Path(wav).stem + f"_{tag}.csv"), index=False)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

train_df = process_pairs(train_pairs, OUT_TRAIN, "train")
test_df = process_pairs(test_pairs, OUT_TEST, "test")

# =============================================================================
# MODEL TRAINING
# =============================================================================
ignore_cols = ["file", "frameTime", "label", "frameIndex","F0direction","directionScore","spectralFlux_log","spectralCentroid_log",
                "spectralMaxPos_log","spectralMinPos_log","pcm_RMSenergy","pcm_LOGenergy"]
feature_cols = [c for c in train_df.columns if c not in ignore_cols]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test = test_df[feature_cols].values
y_test = test_df["label"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
model.fit(X_train_s, y_train)

# =============================================================================
# PREDICTION + TEMPORAL SMOOTHING
# =============================================================================
test_df["wheeze_prob"] = model.predict_proba(X_test_s)[:, 1]

test_df["wheeze_prob_smooth"] = (
    test_df.groupby("file")["wheeze_prob"]
    .transform(lambda x: median_filter(x, size=SMOOTH_WIN))
)

test_df["predicted_label"] = (test_df["wheeze_prob_smooth"] > 0.5).astype(int)

# =============================================================================
# SAVE PER-FILE RESULTS
# =============================================================================
for fname, df_f in test_df.groupby("file"):
    df_f.to_csv(
        os.path.join(OUT_TEST, Path(fname).stem + "_pred.csv"),
        index=False
    )

# =============================================================================
# METRICS
# =============================================================================
print("\nFRAME-LEVEL METRICS (SMOOTHED)")
print(classification_report(
    y_test,
    test_df["predicted_label"],
    target_names=["Normal", "Wheeze"]
))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE")

Found 100 valid .wav/.txt pairs

Processing 80 train files...


100%|██████████| 80/80 [00:11<00:00,  7.10it/s]



Processing 20 test files...


100%|██████████| 20/20 [00:02<00:00,  7.31it/s]



FRAME-LEVEL METRICS (SMOOTHED)
              precision    recall  f1-score   support

      Normal       0.24      0.02      0.03     11781
      Wheeze       0.60      0.96      0.74     18039

    accuracy                           0.59     29820
   macro avg       0.42      0.49      0.39     29820
weighted avg       0.46      0.59      0.46     29820

Confusion Matrix:
 [[  208 11573]
 [  645 17394]]

DONE


In [17]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa
from scipy.signal import find_peaks
from scipy.ndimage import median_filter


# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train_voicing"
OUT_TEST = "csv_test_voicing"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)

SMOOTH_WIN = 7  # frames (~70 ms)

SR = 16000
WIN_DUR = 0.100   # 100 ms
HOP_DUR = 0.010   # 10 ms
FREQ_MAX = 2000


# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)


# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels


def wheeze_priority_label(labels, t_start, t_end):
    """Return 1 if ANY wheeze overlaps window"""
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0


def fill_short_zero_gaps(labels, max_gap=4):
    """
    Convert runs of 0s to 1s if they are between two 1s
    and length <= max_gap
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i  # first non-zero

            # Check bounded by ones
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1

    return labels


# =============================================================================
# VOICING FEATURES
# =============================================================================
def extract_voicing_features(frame, sr):
    """Extract voicing features from audio frame"""
    # 1. Autocorrelation-based F0 (fundamental frequency)
    frame_ac = np.correlate(frame, frame, mode='full')[len(frame)-1:]
    peaks, _ = find_peaks(frame_ac, height=0.3*np.max(frame_ac))
    if len(peaks) > 1:
        peak_diffs = peaks[1:] - peaks[:-1]
        f0_candidates = sr / peak_diffs
        valid_f0 = f0_candidates[(f0_candidates > 50) & (f0_candidates < 1000)]
        f0 = np.median(valid_f0) if len(valid_f0) > 0 else 0
    else:
        f0 = 0
    
    # 2. Harmonic-to-Noise Ratio (HNR)
    fft_mag = np.abs(np.fft.rfft(frame))
    freqs = np.fft.rfftfreq(len(frame), 1/sr)
    harm_peaks = find_peaks(fft_mag, distance=sr/200)[0]  # ~5ms min spacing
    harm_energy = np.sum(fft_mag[harm_peaks]) if len(harm_peaks)>0 else 0
    total_energy = np.sum(fft_mag) + 1e-12
    hnr = 10*np.log10(harm_energy / total_energy)
    
    # 3. Zero Crossing Rate (unvoiced indicator)
    zcr = np.mean(np.diff(np.signbit(frame)) != 0)
    
    # 4. Spectral rolloff (voicing energy concentration)
    cum_energy = np.cumsum(fft_mag)
    total_energy = cum_energy[-1]
    rolloff_idx = np.argmax(cum_energy > 0.85 * total_energy)
    rolloff_freq = freqs[rolloff_idx] if rolloff_idx < len(freqs) else freqs[-1]
    
    return f0, hnr, zcr, rolloff_freq


# =============================================================================
# FEATURE EXTRACTION (ENHANCED WITH SPECTRAL + VOICING)
# =============================================================================
def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []
    frame_labels = []

    for i, start in enumerate(range(0, len(audio) - win_len + 1, hop_len)):
        end = start + win_len
        frame = audio[start:end]
        frame_win = frame * np.hanning(len(frame))

        # FFT SPECTRAL FEATURES
        X = np.fft.rfft(frame_win)
        freqs = np.fft.rfftfreq(len(frame_win), d=1/SR)
        mag = np.abs(X)
        mag[0] = 0  # remove DC

        valid = freqs <= FREQ_MAX
        freqs = freqs[valid]
        mag = mag[valid]

        # Core spectral features
        idx = np.argmax(mag)
        amp = mag[idx]
        freq = freqs[idx]


        # VOICING FEATURES
        f0, hnr, zcr, rolloff_freq = extract_voicing_features(frame_win, SR)

        t_start = start / SR
        t_end = end / SR
        label = wheeze_priority_label(labels, t_start, t_end)

        rows.append([
            Path(wav_path).name,
            t_start,
            amp, freq,
            f0, hnr, zcr, rolloff_freq  # NEW voicing features
        ])
        frame_labels.append(label)

    # ---- APPLY ZERO-GAP BIAS HERE ----
    frame_labels = np.array(frame_labels)
    frame_labels = fill_short_zero_gaps(frame_labels, max_gap=4)

    df = pd.DataFrame(rows, columns=[
        "file", "time_step_s",
        "amplitude", "frequency",
        "f0", "hnr", "zcr", "rolloff_freq"
    ])
    df["label"] = frame_labels

    return df


# =============================================================================
# PROCESS TRAIN FILES
# =============================================================================
train_frames = []
for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)


# =============================================================================
# PROCESS TEST FILES
# =============================================================================
test_frames = []
for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)


# =============================================================================
# MODEL TRAINING (ALL FEATURES)
# =============================================================================
feature_cols = [
    "amplitude", "frequency", "f0", "hnr", "zcr", "rolloff_freq"
]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test = test_df[feature_cols].values
y_test = test_df["label"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Class imbalance weight
n_pos = np.sum(y_train == 1)  # Wheeze frames
n_neg = np.sum(y_train == 0)  # Normal frames
scale_pos_weight = n_neg / n_pos
print(f"scale_pos_weight (Wheeze) = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train_s, y_train)


# =============================================================================
# PREDICTION + TEMPORAL SMOOTHING + SAVE TEST CSVs
# =============================================================================
test_df["wheeze_prob"] = model.predict_proba(X_test_s)[:, 1]

# Temporal smoothing per file
test_df["wheeze_prob_smooth"] = (
    test_df.groupby("file")["wheeze_prob"]
    .transform(lambda x: median_filter(x, size=SMOOTH_WIN))
)

# Final decision
test_df["predicted_label"] = (test_df["wheeze_prob_smooth"] > 0.3).astype(int)

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)


# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                          target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE")
print(f"Train CSVs → {OUT_TRAIN}/")
print(f"Test CSVs  → {OUT_TEST}/")
print(f"Features used: {feature_cols}")


scale_pos_weight (Wheeze) = 0.69
              precision    recall  f1-score   support

      Normal       0.69      0.45      0.55      9266
      Wheeze       0.79      0.91      0.84     20554

    accuracy                           0.77     29820
   macro avg       0.74      0.68      0.69     29820
weighted avg       0.76      0.77      0.75     29820

Confusion Matrix:
 [[ 4215  5051]
 [ 1923 18631]]

DONE
Train CSVs → csv_train_voicing/
Test CSVs  → csv_test_voicing/
Features used: ['amplitude', 'frequency', 'f0', 'hnr', 'zcr', 'rolloff_freq']
